# ASL Recognition Complete Notebook

## 1. Install Dependencies & Mount Drive

In [ ]:
# Cell 1: Setup Environment
!pip install mediapipe tensorflow opencv-python-headless tqdm pandas matplotlib seaborn

import os
from google.colab import drive

# Mount ke Drive biar bisa akses dataset
drive.mount('/content/drive')

# Bikin folder buat nyimpen hasil
os.makedirs('saved_models', exist_ok=True)
os.makedirs('saved_generate', exist_ok=True)

print("Environment siap tempur, ngab! 🔥")

## 2. Path & Constants

In [ ]:
# Cell 2: Konfigurasi
import numpy as np

CLASS_NAMES = [
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J',
    'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T',
    'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space'
]
NUM_CLASSES = len(CLASS_NAMES)
INPUT_DIM = 126 

# CONTOH PATH: /content/drive/MyDrive/Project/dataset/bisindo
DATASET_PATH = "/content/drive/MyDrive/path/ke/dataset/lu"
NPZ_PATH = 'saved_models/landmarks_train.npz'

# Simpan urutan kelas buat dipake di web nanti
np.save('saved_models/landmark_classifier_classes.npy', CLASS_NAMES)

print(f"Dataset Path: {DATASET_PATH}")
print(f"Total Kelas: {NUM_CLASSES}")

## 3. Hand Detector & Smart Extraction

In [ ]:
# Cell 3: Hand Detection & Feature Extraction
import cv2
import mediapipe as mp
from tqdm.notebook import tqdm

class HandDetector:
    def __init__(self):
        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands(static_image_mode=True, max_num_hands=2, min_detection_confidence=0.5)
        
    def detect(self, image):
        rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = self.hands.process(rgb)
        detected_hands = []
        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                lm = np.array([[l.x, l.y, l.z] for l in hand_landmarks.landmark])
                detected_hands.append(lm)
        return detected_hands

def extract_from_scratch(data_dir):
    detector = HandDetector()
    X, y = [], []
    active_classes = [c for c in CLASS_NAMES if os.path.exists(os.path.join(data_dir, c))]
    
    for idx, name in enumerate(tqdm(active_classes, desc="Processing Classes")):
        class_dir = os.path.join(data_dir, name)
        for img_name in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_name)
            img = cv2.imread(img_path)
            if img is None: continue
            hands = detector.detect(img)
            if hands:
                hands = sorted(hands, key=lambda h: h[0][0])
                combined = np.zeros(INPUT_DIM)
                base_wrist = hands[0][0].copy()
                all_norms = [h - base_wrist for h in hands[:2]]
                max_dist = max([np.max(np.linalg.norm(n, axis=1)) for n in all_norms])
                for i, norm in enumerate(all_norms):
                    if max_dist > 0: norm = norm / max_dist
                    combined[i*63:(i+1)*63] = norm.flatten()
                X.append(combined)
                y.append(CLASS_NAMES.index(name))
    return np.array(X), np.array(y)

# Logic Load/Extract
if os.path.exists(NPZ_PATH):
    print("File .npz Ditemukan...")
    data = np.load(NPZ_PATH)
    X_data, y_data = data['X'], data['y']
else:
    print("Belum ada .npz, mulai ekstrak dari Drive...")
    X_data, y_data = extract_from_scratch(DATASET_PATH)
    np.savez(NPZ_PATH, X=X_data, y=y_data)

print(f"Data ready: {len(X_data)} sampel.")

## 4. Model Architecture Dual-Engine (MobileNet & EfficientNet style)

In [ ]:
# Cell 4: Model Architecture
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras import Model

def make_mobilenet_engine():
    inputs = Input(shape=(INPUT_DIM,))
    x = Dense(256, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    return Model(inputs, Dense(NUM_CLASSES, activation='softmax')(x), name="Engine_Fast")

def make_efficientnet_engine():
    inputs = Input(shape=(INPUT_DIM,))
    x = Dense(512, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    return Model(inputs, Dense(NUM_CLASSES, activation='softmax')(x), name="Engine_Accurate")

## 5. Training & Evaluasi (Generate PNG ke Folder)

In [ ]:
# Cell 5: Training & Metrics Generation
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

X_train, X_val, y_train, y_val = train_test_split(X_data, y_data, test_size=0.2, random_state=42)
y_train_cat = tf.keras.utils.to_categorical(y_train, NUM_CLASSES)
y_val_cat = tf.keras.utils.to_categorical(y_val, NUM_CLASSES)

def train_and_eval(model, epochs, save_name):
    print(f"\n🚀 Training {model.name}...")
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(X_train, y_train_cat, validation_data=(X_val, y_val_cat), epochs=epochs, batch_size=32, verbose=1)
    model.save(f'saved_models/{save_name}.keras')
    
    # Generate Metrics
    y_pred = np.argmax(model.predict(X_val), axis=1)
    
    # Classification Report PNG
    report = classification_report(y_val, y_pred, target_names=CLASS_NAMES, output_dict=True)
    sns.heatmap(pd.DataFrame(report).iloc[:-1, :].T, annot=True, cmap="YlGnBu")
    plt.savefig(f'saved_generate/report_{save_name}.png')
    plt.show()
    plt.close()

# Run All Training
train_and_eval(make_mobilenet_engine(), 40, 'mobilenet_landmark')
train_and_eval(make_efficientnet_engine(), 60, 'efficientnet_landmark')

## 6. Hand Detection (MediaPipe)

In [10]:
from dataclasses import dataclass
from typing import List, Tuple
import os

@dataclass
class HandResult:
    landmarks: np.ndarray
    bbox: Tuple[int, int, int, int]
    handedness: str
    confidence: float


class HandDetector:
    """Hand detection using MediaPipe (compatible with 0.10.x)."""
    
    def __init__(self, max_num_hands=2, min_detection_confidence=0.7):
        # Try legacy API first, then new API
        try:
            self.mp_hands = mp.solutions.hands
            self.mp_drawing = mp.solutions.drawing_utils
            self.mp_styles = mp.solutions.drawing_styles
            self.hands = self.mp_hands.Hands(
                static_image_mode=False, max_num_hands=max_num_hands,
                min_detection_confidence=min_detection_confidence
            )
            self.use_legacy = True
        except AttributeError:
            # MediaPipe 0.10.x+ uses Tasks API
            from mediapipe.tasks import python
            from mediapipe.tasks.python import vision
            
            model_path = 'hand_landmarker.task'
            if not os.path.exists(model_path):
                import urllib.request
                url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
                print("Downloading hand landmarker model...")
                urllib.request.urlretrieve(url, model_path)
            
            base_options = python.BaseOptions(model_asset_path=model_path)
            options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=max_num_hands)
            self.detector = vision.HandLandmarker.create_from_options(options)
            self.use_legacy = False
    
    def detect(self, frame, draw=False):
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        hands = []
        h, w = frame.shape[:2]
        
        if self.use_legacy:
            results = self.hands.process(rgb)
            if results.multi_hand_landmarks:
                for idx, hand_lm in enumerate(results.multi_hand_landmarks):
                    lm = np.array([[l.x, l.y, l.z] for l in hand_lm.landmark], dtype=np.float32)
                    x_coords = (lm[:, 0] * w).astype(int)
                    y_coords = (lm[:, 1] * h).astype(int)
                    pad = 40
                    x, y = max(0, x_coords.min()-pad), max(0, y_coords.min()-pad)
                    x2, y2 = min(w, x_coords.max()+pad), min(h, y_coords.max()+pad)
                    handedness = results.multi_handedness[idx].classification[0].label if results.multi_handedness else 'Right'
                    conf = results.multi_handedness[idx].classification[0].score if results.multi_handedness else 0.0
                    hands.append(HandResult(lm, (x, y, x2-x, y2-y), handedness, conf))
        else:
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            results = self.detector.detect(mp_image)
            if results.hand_landmarks:
                for idx, hand_lm in enumerate(results.hand_landmarks):
                    lm = np.array([[l.x, l.y, l.z] for l in hand_lm], dtype=np.float32)
                    x_coords = (lm[:, 0] * w).astype(int)
                    y_coords = (lm[:, 1] * h).astype(int)
                    pad = 40
                    x, y = max(0, x_coords.min()-pad), max(0, y_coords.min()-pad)
                    x2, y2 = min(w, x_coords.max()+pad), min(h, y_coords.max()+pad)
                    handedness = results.handedness[idx][0].category_name if results.handedness else 'Right'
                    conf = results.handedness[idx][0].score if results.handedness else 0.0
                    hands.append(HandResult(lm, (x, y, x2-x, y2-y), handedness, conf))
        return hands
    
    def release(self):
        if self.use_legacy and hasattr(self, 'hands'):
            self.hands.close()

print("HandDetector class defined (compatible with MediaPipe 0.10.x)!")

HandDetector class defined (compatible with MediaPipe 0.10.x)!


## 7. Extract Landmarks from Dataset

In [11]:
def extract_landmarks_from_dataset(data_dir, max_per_class=100):
    """Extract hand landmarks from dataset images."""
    detector = HandDetector(max_num_hands=1)
    all_landmarks = []
    all_labels = []
    
    available_classes = [c for c in CLASS_NAMES if os.path.exists(os.path.join(data_dir, c))]
    
    for class_idx, class_name in enumerate(tqdm(available_classes)):
        class_dir = os.path.join(data_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
        
        count = 0
        for img_name in os.listdir(class_dir):
            if count >= max_per_class:
                break
            img_path = os.path.join(class_dir, img_name)
            try:
                img = cv2.imread(img_path)
                if img is None:
                    continue
                hands = detector.detect(img, draw=False)
                if hands:
                    lm = hands[0].landmarks
                    # Normalize
                    wrist = lm[0].copy()
                    normalized = lm - wrist
                    max_dist = np.max(np.linalg.norm(normalized, axis=1))
                    if max_dist > 0:
                        normalized = normalized / max_dist
                    all_landmarks.append(normalized.flatten())
                    all_labels.append(class_idx)
                    count += 1
            except:
                continue
    
    detector.release()
    return np.array(all_landmarks), np.array(all_labels)

# Run extraction if dataset exists
if os.path.exists(TRAIN_DIR):
    print("Extracting landmarks...")
    X_landmarks, y_landmarks = extract_landmarks_from_dataset(TRAIN_DIR, max_per_class=100)
    print(f"Extracted {len(X_landmarks)} landmark samples")
    np.savez('saved_models/landmarks_train.npz', X=X_landmarks, y=y_landmarks)
else:
    print("Dataset not found. Skip landmark extraction.")

Extracting landmarks...


OSError: [Errno 28] No space left on device

## 8. Train Landmark Classifier

In [ ]:
# Load or use extracted landmarks
if os.path.exists('saved_models/landmarks_train.npz'):
    data = np.load('saved_models/landmarks_train.npz')
    X = data['X']
    y = data['y']
    print(f"Loaded {len(X)} landmark samples")
    
    # Split data
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    y_train_cat = to_categorical(y_train, NUM_CLASSES)
    y_val_cat = to_categorical(y_val, NUM_CLASSES)
    
    # Create and train model
    landmark_model = create_landmark_model()
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ModelCheckpoint(LANDMARK_MODEL_PATH, monitor='val_accuracy', save_best_only=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
    ]
    
    history_lm = landmark_model.fit(
        X_train, y_train_cat,
        validation_data=(X_val, y_val_cat),
        epochs=50,
        batch_size=32,
        callbacks=callbacks
    )
    
    # Save class names
    np.save('saved_models/landmark_classifier_classes.npy', CLASS_NAMES)
    print(f"Landmark model saved to: {LANDMARK_MODEL_PATH}")
else:
    print("Landmarks not found. Run extraction first.")

## 9. Evaluation & Visualization

In [ ]:
def plot_training_history(history, title='Training History'):
    """Plot training history."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(history.history['accuracy'], label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Val')
    axes[0].set_title(f'{title} - Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True)
    
    axes[1].plot(history.history['loss'], label='Train')
    axes[1].plot(history.history['val_loss'], label='Val')
    axes[1].set_title(f'{title} - Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150)
    plt.show()


def plot_confusion_matrix(y_true, y_pred, classes):
    """Plot confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(15, 12))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes)
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=150)
    plt.show()


# Plot history if available
if 'history' in dir():
    plot_training_history(history, 'CNN Model')
if 'history_lm' in dir():
    plot_training_history(history_lm, 'Landmark Model')

In [ ]:
# Evaluate landmark model
if os.path.exists(LANDMARK_MODEL_PATH) and 'X_val' in dir():
    model_eval = tf.keras.models.load_model(LANDMARK_MODEL_PATH)
    predictions = model_eval.predict(X_val)
    y_pred = np.argmax(predictions, axis=1)
    
    # Classification report
    unique_classes = np.unique(y_val)
    class_labels = [CLASS_NAMES[i] for i in unique_classes]
    print(classification_report(y_val, y_pred, target_names=class_labels))
    
    # Confusion matrix
    plot_confusion_matrix(y_val, y_pred, class_labels)
    
    accuracy = np.sum(y_pred == y_val) / len(y_val)
    print(f"\nOverall Accuracy: {accuracy:.4f}")

## 10. Test Inference (Real-time Demo)

In [ ]:
class LandmarkClassifier:
    """Landmark-based ASL classifier."""
    
    def __init__(self, model_path=LANDMARK_MODEL_PATH):
        self.model = None
        self.class_names = CLASS_NAMES
        if os.path.exists(model_path):
            self.model = tf.keras.models.load_model(model_path)
            print(f"Model loaded: {model_path}")
    
    def normalize(self, landmarks):
        if landmarks is None:
            return None
        normalized = landmarks - landmarks[0]
        max_dist = np.max(np.linalg.norm(normalized, axis=1))
        if max_dist > 0:
            normalized = normalized / max_dist
        return normalized
    
    def predict(self, landmarks):
        if self.model is None or landmarks is None:
            return '?', 0.0
        normalized = self.normalize(landmarks)
        flat = normalized.flatten().reshape(1, -1)
        pred = self.model.predict(flat, verbose=0)[0]
        idx = np.argmax(pred)
        return self.class_names[idx], float(pred[idx])

# Test
if os.path.exists(LANDMARK_MODEL_PATH):
    classifier = LandmarkClassifier()
    print("Classifier ready for inference!")

In [ ]:
# Real-time demo (uncomment to run)
# WARNING: This will open your webcam

# def run_demo():
#     detector = HandDetector(max_num_hands=1)
#     classifier = LandmarkClassifier()
#     cap = cv2.VideoCapture(0)
#     
#     while cap.isOpened():
#         ret, frame = cap.read()
#         if not ret:
#             break
#         frame = cv2.flip(frame, 1)
#         hands = detector.detect(frame, draw=True)
#         
#         if hands:
#             letter, conf = classifier.predict(hands[0].landmarks)
#             cv2.putText(frame, f'{letter}: {conf:.0%}', (10, 50),
#                        cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)
#         
#         cv2.imshow('ASL Recognition', frame)
#         if cv2.waitKey(1) & 0xFF == ord('q'):
#             break
#     
#     cap.release()
#     cv2.destroyAllWindows()
#     detector.release()

# run_demo()

---
## Summary

Notebook ini berisi:
1. ✅ Data loading & preprocessing
2. ✅ CNN model architecture (MobileNetV2)
3. ✅ Training pipeline
4. ✅ Hand detection (MediaPipe)
5. ✅ Landmark extraction & training
6. ✅ Evaluation & visualization
7. ✅ Real-time inference demo

**Untuk menjalankan training:**
1. Tambahkan dataset ke `dataset/asl_alphabet_train/`
2. Jalankan semua cells dari atas ke bawah